In [8]:
import pandas as pd
import numpy as np

data = {
    "customer_id": [1, 2, 3, 4, 5, 1, 2, 3, 4, 5, 10, 11, 12, 13, 14],
    "age": [25, 30, 35, 40, 45, 25, 30, 35, 40, 45, 50, -5, 29, 33, 41],
    "income": [
        50000, 60000, 55000, 58000, 62000,
        50000, 60000, 55000, 58000, 62000,
        999999999, 45000, np.nan, np.nan, np.nan
    ],
    "purchases": [3, 4, 2, 5, 6, 3, 4, 2, 5, 6, 10, 2, "ABC", 4, 5]
}

df = pd.DataFrame(data)

print("Dataset:")
print(df)

print("\nDataset Shape:")
print(df.shape)

print("\nMissing Values:")
print(df.isnull().sum())

print("Duplicate Rows:")
print(df.duplicated().sum())

print("\nData Types:")
print(df.dtypes)

Dataset:
    customer_id  age       income purchases
0             1   25      50000.0         3
1             2   30      60000.0         4
2             3   35      55000.0         2
3             4   40      58000.0         5
4             5   45      62000.0         6
5             1   25      50000.0         3
6             2   30      60000.0         4
7             3   35      55000.0         2
8             4   40      58000.0         5
9             5   45      62000.0         6
10           10   50  999999999.0        10
11           11   -5      45000.0         2
12           12   29          NaN       ABC
13           13   33          NaN         4
14           14   41          NaN         5

Dataset Shape:
(15, 4)

Missing Values:
customer_id    0
age            0
income         3
purchases      0
dtype: int64
Duplicate Rows:
5

Data Types:
customer_id      int64
age              int64
income         float64
purchases       object
dtype: object


In [9]:
def quality_report(dataframe):
    report = {}

    # 1. Dataset shape
    report["rows"] = dataframe.shape[0]
    report["columns"] = dataframe.shape[1]

    # 2. Missing values
    report["missing_values"] = dataframe.isnull().sum().to_dict()

    # 3. Duplicate rows
    report["duplicate_rows"] = dataframe.duplicated().sum()

    # 4. Data types
    report["data_types"] = dataframe.dtypes.astype(str).to_dict()

    # 5. Invalid age check
    invalid_age_count = dataframe[dataframe["age"] < 0].shape[0]
    report["invalid_age_count"] = invalid_age_count

    # 6. Invalid purchases check
    purchases_numeric = pd.to_numeric(dataframe["purchases"], errors="coerce")

    invalid_purchases_count = (
        purchases_numeric.isnull().sum()
        - dataframe["purchases"].isnull().sum()
    )

    report["invalid_purchases_count"] = invalid_purchases_count

    # 7. Income outlier check using IQR
    income_clean = dataframe["income"].dropna()

    Q1 = income_clean.quantile(0.25)
    Q3 = income_clean.quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    income_outliers_count = dataframe[
        (dataframe["income"] < lower_bound) |
        (dataframe["income"] > upper_bound)
    ].shape[0]

    report["income_outliers_count"] = income_outliers_count

    # 8. Total issues
    total_issues = (
        dataframe.isnull().sum().sum()
        + dataframe.duplicated().sum()
        + invalid_age_count
        + invalid_purchases_count
        + income_outliers_count
    )

    report["total_issues"] = int(total_issues)

    return report

In [10]:
before_report = quality_report(df)

print("\nQuality Report Before Cleaning:")
print(before_report)


Quality Report Before Cleaning:
{'rows': 15, 'columns': 4, 'missing_values': {'customer_id': 0, 'age': 0, 'income': 3, 'purchases': 0}, 'duplicate_rows': np.int64(5), 'data_types': {'customer_id': 'int64', 'age': 'int64', 'income': 'float64', 'purchases': 'object'}, 'invalid_age_count': 1, 'invalid_purchases_count': np.int64(1), 'income_outliers_count': 1, 'total_issues': 11}


In [11]:
def clean_data(dataframe):
    df_clean = dataframe.copy()

    # 1. Remove duplicate rows
    df_clean = df_clean.drop_duplicates()

    # 2. Remove invalid age values
    df_clean = df_clean[df_clean["age"] >= 0]

    # 3. Convert purchases column to numeric
    df_clean["purchases"] = pd.to_numeric(df_clean["purchases"], errors="coerce")

    # 4. Drop rows where purchases became NaN
    df_clean = df_clean.dropna(subset=["purchases"])

    # 5. Handle income outlier using IQR clipping
    Q1 = df_clean["income"].quantile(0.25)
    Q3 = df_clean["income"].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    df_clean["income"] = df_clean["income"].clip(
        lower=lower_bound,
        upper=upper_bound
    )

    # 6. Fill missing income using median
    df_clean["income"] = df_clean["income"].fillna(df_clean["income"].median())

    return df_clean

In [12]:
cleaned_df = clean_data(df)

print("\nCleaned Dataset:")
print(cleaned_df)

after_report = quality_report(cleaned_df)

print("\nQuality Report After Cleaning:")
print(after_report)


Cleaned Dataset:
    customer_id  age   income  purchases
0             1   25  50000.0        3.0
1             2   30  60000.0        4.0
2             3   35  55000.0        2.0
3             4   40  58000.0        5.0
4             5   45  62000.0        6.0
10           10   50  70125.0       10.0
13           13   33  59000.0        4.0
14           14   41  59000.0        5.0

Quality Report After Cleaning:
{'rows': 8, 'columns': 4, 'missing_values': {'customer_id': 0, 'age': 0, 'income': 0, 'purchases': 0}, 'duplicate_rows': np.int64(0), 'data_types': {'customer_id': 'int64', 'age': 'int64', 'income': 'float64', 'purchases': 'float64'}, 'invalid_age_count': 0, 'invalid_purchases_count': np.int64(0), 'income_outliers_count': 2, 'total_issues': 2}


In [13]:
import pandas as pd
import numpy as np


# -------------------------------------------------
# 1. Create Dataset with Data Quality Problems
# -------------------------------------------------

data = {
    "customer_id": [1, 2, 3, 4, 5, 1, 2, 3, 4, 5, 10, 11, 12, 13, 14],
    "age": [25, 30, 35, 40, 45, 25, 30, 35, 40, 45, 50, -5, 29, 33, 41],
    "income": [
        50000, 60000, 55000, 58000, 62000,
        50000, 60000, 55000, 58000, 62000,
        999999999, 45000, np.nan, np.nan, np.nan
    ],
    "purchases": [3, 4, 2, 5, 6, 3, 4, 2, 5, 6, 10, 2, "ABC", 4, 5]
}

df = pd.DataFrame(data)

# Save raw dataset
df.to_csv("raw_customer_data.csv", index=False)


# -------------------------------------------------
# 2. Income Outlier Detection Function
# -------------------------------------------------

def detect_income_outliers(dataframe):
    income_numeric = pd.to_numeric(dataframe["income"], errors="coerce")
    income_clean = income_numeric.dropna()

    if income_clean.empty:
        return 0, None, None

    Q1 = income_clean.quantile(0.25)
    Q3 = income_clean.quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outlier_mask = (income_numeric < lower_bound) | (income_numeric > upper_bound)
    outlier_count = int(outlier_mask.sum())

    return outlier_count, lower_bound, upper_bound


# -------------------------------------------------
# 3. Quality Report Function
# -------------------------------------------------

def create_quality_report(dataframe, stage_name):
    missing_counts = dataframe.isnull().sum()

    duplicate_rows = int(dataframe.duplicated().sum())

    age_numeric = pd.to_numeric(dataframe["age"], errors="coerce")
    invalid_age_count = int((age_numeric < 0).sum())

    purchases_numeric = pd.to_numeric(dataframe["purchases"], errors="coerce")
    invalid_purchases_count = int(
        purchases_numeric.isnull().sum() - dataframe["purchases"].isnull().sum()
    )

    income_outlier_count, lower_bound, upper_bound = detect_income_outliers(dataframe)

    total_issues = int(
        missing_counts.sum()
        + duplicate_rows
        + invalid_age_count
        + invalid_purchases_count
        + income_outlier_count
    )

    report_rows = [
        {
            "Stage": stage_name,
            "Check": "Total Rows",
            "Value": dataframe.shape[0],
            "Details": "Number of rows in dataset"
        },
        {
            "Stage": stage_name,
            "Check": "Total Columns",
            "Value": dataframe.shape[1],
            "Details": "Number of columns in dataset"
        },
        {
            "Stage": stage_name,
            "Check": "Missing customer_id",
            "Value": int(missing_counts["customer_id"]),
            "Details": "Missing values in customer_id column"
        },
        {
            "Stage": stage_name,
            "Check": "Missing age",
            "Value": int(missing_counts["age"]),
            "Details": "Missing values in age column"
        },
        {
            "Stage": stage_name,
            "Check": "Missing income",
            "Value": int(missing_counts["income"]),
            "Details": "Missing values in income column"
        },
        {
            "Stage": stage_name,
            "Check": "Missing purchases",
            "Value": int(missing_counts["purchases"]),
            "Details": "Missing values in purchases column"
        },
        {
            "Stage": stage_name,
            "Check": "Duplicate Rows",
            "Value": duplicate_rows,
            "Details": "Exact repeated rows"
        },
        {
            "Stage": stage_name,
            "Check": "Invalid Age Count",
            "Value": invalid_age_count,
            "Details": "Age values less than 0"
        },
        {
            "Stage": stage_name,
            "Check": "Invalid Purchases Count",
            "Value": invalid_purchases_count,
            "Details": "Non-numeric purchase values like ABC"
        },
        {
            "Stage": stage_name,
            "Check": "Income Outliers Count",
            "Value": income_outlier_count,
            "Details": f"IQR bounds: {lower_bound} to {upper_bound}"
        },
        {
            "Stage": stage_name,
            "Check": "Total Issues",
            "Value": total_issues,
            "Details": "Total detected data quality issues"
        }
    ]

    report_df = pd.DataFrame(report_rows)
    return report_df


# -------------------------------------------------
# 4. Cleaning Function
# -------------------------------------------------

def clean_data(dataframe):
    df_clean = dataframe.copy()

    # Remove duplicate rows
    df_clean = df_clean.drop_duplicates()

    # Remove invalid age values
    df_clean = df_clean[df_clean["age"] >= 0]

    # Convert purchases to numeric
    df_clean["purchases"] = pd.to_numeric(df_clean["purchases"], errors="coerce")

    # Drop rows where purchases is invalid
    df_clean = df_clean.dropna(subset=["purchases"])

    # Handle income outliers using IQR clipping
    income_numeric = pd.to_numeric(df_clean["income"], errors="coerce")

    Q1 = income_numeric.dropna().quantile(0.25)
    Q3 = income_numeric.dropna().quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    df_clean["income"] = df_clean["income"].clip(
        lower=lower_bound,
        upper=upper_bound
    )

    # Fill missing income using median
    df_clean["income"] = df_clean["income"].fillna(df_clean["income"].median())

    return df_clean


# -------------------------------------------------
# 5. Generate Before Quality Report
# -------------------------------------------------

before_report = create_quality_report(df, "Before Cleaning")
before_report.to_csv("quality_report_before.csv", index=False)

print("\nQUALITY REPORT BEFORE CLEANING")
print(before_report)


# -------------------------------------------------
# 6. Clean Dataset
# -------------------------------------------------

cleaned_df = clean_data(df)
cleaned_df.to_csv("clean_customer_data.csv", index=False)

print("\nCLEANED DATASET")
print(cleaned_df)


# -------------------------------------------------
# 7. Generate After Quality Report
# -------------------------------------------------

after_report = create_quality_report(cleaned_df, "After Cleaning")
after_report.to_csv("quality_report_after.csv", index=False)

print("\nQUALITY REPORT AFTER CLEANING")
print(after_report)


# -------------------------------------------------
# 8. Final Confirmation
# -------------------------------------------------

print("\nFiles generated successfully:")
print("1. raw_customer_data.csv")
print("2. clean_customer_data.csv")
print("3. quality_report_before.csv")
print("4. quality_report_after.csv")


QUALITY REPORT BEFORE CLEANING
              Stage                    Check  Value  \
0   Before Cleaning               Total Rows     15   
1   Before Cleaning            Total Columns      4   
2   Before Cleaning      Missing customer_id      0   
3   Before Cleaning              Missing age      0   
4   Before Cleaning           Missing income      3   
5   Before Cleaning        Missing purchases      0   
6   Before Cleaning           Duplicate Rows      5   
7   Before Cleaning        Invalid Age Count      1   
8   Before Cleaning  Invalid Purchases Count      1   
9   Before Cleaning    Income Outliers Count      1   
10  Before Cleaning             Total Issues     11   

                                 Details  
0              Number of rows in dataset  
1           Number of columns in dataset  
2   Missing values in customer_id column  
3           Missing values in age column  
4        Missing values in income column  
5     Missing values in purchases column  
6     